In [7]:
# 1. Imports and setup
from romansh_lemmatizer import Lemmatizer, Lemma
from collections import Counter
import pandas as pd
from pathlib import Path
from datasets import load_dataset
import re
from collections import Counter

mediomatix = load_dataset("ZurichNLP/mediomatix", split="train[:10000]")
idioms = ["rm-sursilv", "rm-sutsilv", "rm-surmiran", "rm-puter", "rm-vallader"]


In [8]:
# 2. Inspection
"""
inspection_data = mediomatix[0]
for k, v in inspection_data.items():
    print(k, v)
print()
"""

'\ninspection_data = mediomatix[0]\nfor k, v in inspection_data.items():\n    print(k, v)\nprint()\n'

In [9]:

# 3. Build corpus
corpus = {k:[] for k in idioms}
for row in mediomatix:
    for idiom in idioms:
        if row.get(idiom):
            sent = row[idiom]
            sent = re.sub(r"</?strong>", " ", sent) # Remove <strong> tags
            sent = re.sub(r'["“”«»‚‘’„][^"“”«»‚‘’„]+["“”«»‚‘’„]', " ", sent) # Remove quoted substrings (straight, angled, curly quotes)
            corpus[idiom].append(sent)

for k, v in corpus.items():
    print(k, len(v))

rm-sursilv 9759
rm-sutsilv 9334
rm-surmiran 4119
rm-puter 9391
rm-vallader 9368


In [10]:
# 4. Initialize lemmatizers (specific idioms)
lemmatizers = {idiom: Lemmatizer(idiom=idiom) for idiom in idioms}

In [11]:
from collections import defaultdict, Counter

corpus_lemma_counter = {
    k: defaultdict(lambda: {"count": 0, "forms": Counter()}) for k in idioms
}

for idiom, sentences in corpus.items():
    for sent in sentences:
        doc = lemmatizers[idiom](sent)
        for t in doc.tokens:
            if any(c in t.text for c in ["/", ".", "-", ":", "(", ")"]):
                continue

            # Use only the first lemma (if any)
            if not t.lemmas:
                continue
            first_lemma = next(iter(t.lemmas.keys()))

            lemma_text = first_lemma.text
            translation = first_lemma.translation_de or ""
            entry = corpus_lemma_counter[idiom][(lemma_text, translation)]

            entry["count"] += 1
            entry["forms"][t.text] += 1


In [12]:
# 6. Pretty print results
# Later when creating the DataFrame:
for idiom, counter in corpus_lemma_counter.items():
   
    # Prepare data rows including the forms with counts formatted like "form (count)"
    rows = []
    for (lemma_text, trans), data in sorted(counter.items(), key=lambda x: x[1]["count"], reverse=True)[:20]:
        forms_str = ", ".join(f"{form} ({count})" for form, count in data["forms"].items())
        rows.append((lemma_text, trans or "", data["count"], forms_str))
    
    df = pd.DataFrame(rows, columns=["Lemma", "German", "Freq", "Forms"])
    print(f"\n===== {idiom} =====")
    import pandas as pd

    # Show full content of cells, but limit visible width (characters)
    pd.set_option('display.max_colwidth', 100)  # e.g. 200 chars per cell

    # Show more rows/columns before truncation
    pd.set_option('display.max_rows', 100)
    pd.set_option('display.max_columns', 10)


    display(df)


===== rm-sursilv =====


,Lemma,German,Freq,Forms
0,il,der,1989,"il (1564), Il (424), IL (1)"
1,da,aus,1902,"da (1859), Da (41), DA (2)"
2,esser,sein,1868,"ei (1254), seigi (20), ein (244), fuss (31), eis (95), Ei (76), staus (27), essan (11), Eis (34)..."
3,e,fünfter Buchstabe des Alphabets,1803,"e (1726), E (77)"
4,la,die,1661,"la (1379), La (282)"
5,in,eins,1525,"in (1420), In (104), IN (1)"
6,ils,cf. il,1327,"ils (1022), Ils (304), ILS (1)"
7,haver,haben,1193,"giu (107), ha (677), han (203), veva (4), veis (1), Has (52), has (128), hagies (5), Han (4), ha..."
8,che,null,1018,"che (1010), Che (8)"
9,ina,eine,1001,"ina (955), Ina (46)"



===== rm-sutsilv =====


,Lemma,German,Freq,Forms
0,igl,es,3105,"igl (2354), Igl (750), IGL (1)"
1,la,null,2816,"la (2416), La (399), LA (1)"
2,a,und,2384,"a (2250), A (134)"
3,da,von,2193,"da (2155), Da (36), DA (2)"
4,ver,besitzen,1467,"â (662), vaseva (3), vevan (6), veza (40), ve (127), ân (184), vieu (21), vegi (12), veva (40), ..."
5,easser,sich aufhalten,1435,"segi (16), e (943), en (244), eara (28), fuss (26), E (31), earan (19), es (35), eassan (14), st..."
6,egn,Eins,1386,"egn (1288), Egn (98)"
7,igls,null,1245,"igls (932), Igls (312), IGLS (1)"
8,las,null,1191,"las (1095), Las (95), LAS (1)"
9,an,in,1018,"an (920), An (98)"



===== rm-surmiran =====


,Lemma,German,Freq,Forms
0,igl,den,1373,"igl (1139), Igl (233), IGL (1)"
1,la,die,1171,"la (979), La (192)"
2,da,als,968,"Da (18), da (950)"
3,e,und,759,"e (737), E (22)"
4,tgi,als,734,"tgi (665), Tgi (69)"
5,igls,die,694,"igls (584), Igls (110)"
6,esser,liegen,686,"è (415), èn (122), Fiss (1), Èn (1), sung (53), Ist (18), fiss (20), ist (17), È (5), ischan (2)..."
7,veir,haben,638,"ò (281), ast (72), on (84), Ast (31), Vez (4), vegia (8), On (4), gia (20), vess (15), vez (16),..."
8,en,Inn,612,"en (568), En (44)"
9,las,sie,582,"las (476), Las (106)"



===== rm-puter =====


,Lemma,German,Freq,Forms
0,ün,Eins,2589,"ün (1419), üna (1026), Ün (103), Üna (41)"
1,da,mit,2416,"da (2379), Da (35), DA (2)"
2,il,das (vor Konsonant),2085,"il (1622), Il (462), IL (1)"
3,la,das (vor Konsonant),2061,"la (1781), La (279), LA (1)"
4,e,E,1780,"e (1711), E (69)"
5,avair,Lust haben,1511,"ho (782), vains (25), he (92), haun (221), hegia (8), vaiva (59), vais (33), Hest (49), hest (13..."
6,a,A,1419,"a (1212), A (207)"
7,esser,im Wald sein,1149,"es (897), eira (61), füss (32), eiran (23), est (30), Eira (1), essans (10), Est (23), Es (19), ..."
8,ils,die,1147,"ils (831), Ils (315), ILS (1)"
9,las,die,1100,"las (1019), Las (80), LAS (1)"



===== rm-vallader =====


,Lemma,German,Freq,Forms
0,da,über,2432,"da (2396), Da (35), DA (1)"
1,il,das (vor Konsonanten),2033,"il (1583), Il (449), IL (1)"
2,la,La,2018,"la (1733), La (284), LA (1)"
3,e,E,1821,"e (1752), E (69)"
4,ün,ein,1523,"ün (1418), Ün (105)"
5,a,A,1273,"a (1158), A (115)"
6,ils,die,1111,"ils (793), Ils (318)"
7,in,an,1071,"in (984), In (87)"
8,esser,an etw gewöhnt sein,1052,"es (802), eira (66), eiran (24), est (30), füss (27), saja (20), eschan (10), Est (23), eschat (..."
9,üna,eine,1037,"üna (996), Üna (41)"
